# 01 — Data Exploration

Explore raw tick and OHLCV data ingested by `nexus_data`.

**Topics covered:**
- Connect to DuckDB and inspect tables
- Tick data quality — gaps, spread distribution, outliers
- OHLCV candle completeness across M1 → H4
- Session-tagged volume profiles
- Basic descriptive statistics per symbol

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import duckdb
import pandas as pd
import plotly.express as px

DB_PATH = "../data/nexus.duckdb"

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
tables = conn.execute("SHOW TABLES").fetchdf()
display(tables)

In [ ]:
# Tick data sample
ticks = conn.execute(
    "SELECT * FROM ticks WHERE symbol='EURUSD' ORDER BY timestamp DESC LIMIT 1000"
).fetchdf()
print(f"Rows: {len(ticks)}")
ticks.describe()

In [ ]:
# Spread distribution
if not ticks.empty and 'spread' in ticks.columns:
    fig = px.histogram(ticks, x='spread', nbins=50, title='EURUSD Spread Distribution')
    fig.show()

In [ ]:
# OHLCV completeness
ohlcv = conn.execute(
    "SELECT timeframe, COUNT(*) as cnt, MIN(timestamp) as first, MAX(timestamp) as last "
    "FROM ohlcv WHERE symbol='EURUSD' GROUP BY timeframe ORDER BY timeframe"
).fetchdf()
display(ohlcv)

In [ ]:
conn.close()
print("Done.")